# Modal

In [ ]:
# | default_exp run_modal

In [ ]:
# | hide
from nbdev.showdoc import *  # type: ignore

Works for now -- in the future make a standalone fastapi server and run on fly gpu.

In [2]:
import os
import subprocess
import sys
from dotenv import load_dotenv  # type: ignore

if os.getenv("USER") == "max":
    # Get the current working directory
    current_dir = os.getcwd()

    # Append the parent directory to sys.path
    parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
    sys.path.append(parent_dir)

    # Decrypt the .env.local file and load variables
    try:
        # Decrypt and output to stdout
        os.system(
            f"dotenvx decrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local"
        )

        # Load the decrypted environment variables from stdout
        load_dotenv(
            dotenv_path=os.path.abspath(os.path.join(current_dir, "..")) + "/.env.local"
        )

        os.system(
            f"dotenvx encrypt -f {os.path.abspath(os.path.join(current_dir, '..'))}/.env.local"
        )

        print("Environment variables loaded successfully.")
        os.environ["DOCKER_HOST"] = "unix:///Users/max/.docker/run/docker.sock"
    except subprocess.CalledProcessError as e:
        print(f"Error decrypting .env.local: {e}")
        sys.exit(1)

✔ decrypted (/Users/max/Documents/product_horse/.env.local)
✔ encrypted (/Users/max/Documents/product_horse/.env.local)
Environment variables loaded successfully.


In [3]:
# | export
import modal
from product_horse.audio_video import create_video_from_utterances
from product_horse.db import SqlModelDatabase, UtteranceSegment, FileMetadata, Employee, Transcription, UnvalidatedTranscription, UnvalidatedUtterance
from product_horse.audio_video import AudioEditor
from product_horse.filesystems import LocalFileSystem, R2StorageClient
from product_horse.graphql import UtteranceSegmentInput
from uuid import UUID
from random import randint
import uuid
import os
from typing import Tuple, List
from tenacity import retry, stop_after_attempt, wait_exponential

In [47]:
# | export
api_key = os.getenv("ASSEMBLYAI_API_KEY")
if not api_key:
    raise ValueError("ASSEMBLYAI_API_KEY environment variable not found")
audio_editor = AudioEditor(api_key=api_key)


app = modal.App("process-video")
# add -dev if needed
image = (
    modal.Image.debian_slim()
    .apt_install(["ffmpeg", "pkg-config", "libcairo2-dev", "imagemagick", "ghostscript", "fonts-freefont-otf"])
    .pip_install_from_requirements("requirements.lock")
    .run_commands(
        """sed -i 's/rights="none" pattern="@\\*"/rights="read|write" pattern="@*"/' /etc/ImageMagick-6/policy.xml"""
    )
)
# add _dev if needed
@app.function(timeout=1200, secrets=[modal.Secret.from_name("video_secrets")], image=image)
async def run_remote_create_video(
    utterance_segments_inputs: list[UtteranceSegmentInput],
    employee_id: UUID,
    jwt: str,
    title: str,
    size: Tuple[int, int] = (1920, 1080),
    force_size: bool = True,
):
    """
    TO-DOs:
    - recover if file fails to download
    - update video status more immediately and frequently
    - set video failed status if that happens
    """
    db_url = os.getenv("DATABASE_URL")
    db_superuser_url = os.getenv("DATABASE_SUPERUSER_URL")
    if db_superuser_url is None:
        raise Exception("DATABASE_SUPERUSER_URL is not set")
    API_URL = os.getenv("API_URL")
    if db_url is None or API_URL is None:
        raise Exception(
            f"Not Set: {'DATABASE_URL' if db_url is None else ''} {'API_URL' if API_URL is None else ''}"
        )
    database_superuser = SqlModelDatabase(database_url=db_superuser_url)
    database = SqlModelDatabase(database_url=db_url)
    # run on superuser connection, since it's behind jwt auth
    employee = database_superuser.get_employee(employee_id=employee_id)
    if employee is None:
        raise Exception("Employee not found")
    # DEFINE FILE SYSTEMS
    r2 = R2StorageClient(
        api_url=API_URL,
        base_path=str(employee.company_id),
        jwt=jwt,
    )
    server_file_system = LocalFileSystem()
    # rest of program
    utterance_ids = [
        UUID(segment.utterance_id) for segment in utterance_segments_inputs
    ]
    word_ids = [
        word_id
        for segment in utterance_segments_inputs
        for word_id in segment.word_ids
    ]
    utterances = database.as_employee(employee).get_utterances(
        utterance_ids, word_ids=word_ids
    )
    video_exists = database.as_employee(employee).get_all_videos()
    for video in video_exists:
        if video.title == title:  # type: ignore
            title = f"{title}-{randint(1, 200)}"
    utterance_segments_for_video: list[UtteranceSegment] = []
    for utterance in utterances:
        utterance_segments_for_video.append(
            UtteranceSegment(
                id=utterance.id,
                transcription_id=utterance.transcription_id,
                transcription=utterance.transcription,
                words=utterance.words,
                segment_start_word=utterance.words[0],  # type: ignore
                segment_end_word=utterance.words[-1],  # type: ignore
                confidence=utterance.confidence,
                company_id=utterance.company_id,
                created_by_id=utterance.created_by_id,
                start=utterance.start,
                end=utterance.end,
            )
        )
    user = database.as_employee(employee).get_all_users()[0]
    if user is None:
        raise Exception("User not found")
    final_destination = await r2.get_unique_file_key(f"{title}.mp4", str(user.id))
    video = await create_video_from_utterances(
        database,
        remote_file_system=r2,
        local_file_system=server_file_system,
        employee=employee,
        user=user,
        utterance_segments=utterance_segments_for_video,
        final_destination=final_destination,
        title=title,
        size=size,
        force_size=force_size,
    )
    return video



@retry(stop=stop_after_attempt(2), wait=wait_exponential(multiplier=1, min=4, max=15))
async def transcribe_and_save_file(
    file_metadata: FileMetadata,
    signed_url: str,
    employee: Employee,
    db: SqlModelDatabase,
) -> Transcription:
    transcription_result = await audio_editor.transcribe(signed_url, employee)
    utterances: List[UnvalidatedUtterance] = transcription_result.utterances

    unvalidated_transcription = UnvalidatedTranscription(
        file_id=file_metadata.id,
        text=transcription_result.text,
        company_id=employee.company_id,
        created_by_id=employee.id,
    )
    transcription: Transcription = db.as_employee(employee).save_transcription(
        unvalidated_transcription, utterances
    )
    return transcription

@app.function(secrets=[modal.Secret.from_name("video_secrets")], image=image)
async def run_remote_transcribe_and_save_file(
    file_metadata_to_save: FileMetadata,
    employee: Employee,
    jwt: str,
):
    API_URL = os.getenv("API_URL")
    if API_URL is None:
        raise Exception("API_URL is not set")
    print("API_URL", API_URL)
    r2 = R2StorageClient(
        api_url=API_URL,
        base_path=str(employee.company_id),
        jwt=jwt,
    )
    db_url = os.getenv("DATABASE_URL")
    if db_url is None:
        raise Exception("DATABASE_URL is not set")
    database = SqlModelDatabase(database_url=db_url)
    signed_url = r2.get_signed_url(file_metadata_to_save.file_path)
    return await transcribe_and_save_file(
        db=database,
        file_metadata=file_metadata_to_save,
        signed_url=signed_url,
        employee=employee,
    )


In [5]:
test_multiple_inputs = modal.Function.lookup("test-app", "test_multiple_inputs")

# Example inputs
x_list = [{"x": 1, "y": 2}, {"x": 3, "y": 4}]
y_list = ["a", "b", "c"]
z = "example"

result = test_multiple_inputs.remote(x_list, y_list, z)
print(result)

([{'x': 1, 'y': 2}, {'x': 3, 'y': 4}], ['a', 'b', 'c'], 'example')


In [ ]:
job_id = str(uuid.uuid4())
make_video.spawn(
    job_id=job_id,
    params={
        "utterance_segments_inputs": utterance_segments_inputs,
        "employee_id": employee_id,
        "jwt": jwt,
        "title": title,
        "size": size,
        "force_size": force_size,
    },
)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()  # type: ignore